# Chapitre 15 - Forêts aléatoires (1) : définition, algorithme, erreur *out-of-bag*

Ce premier notebook présente les **forêts aléatoires** (*random forests*, Breiman 2001),
sections 15.1 à 15.3.1 du livre :

1. rappel sur le *bagging* et calcul de la variance d'une moyenne de variables corrélées
   (formule (15.1)) ;
2. l'algorithme 15.1 et une implémentation « à la main » à partir d'arbres de régression ;
3. l'application aux données **spam** et l'erreur **out-of-bag** (figure 15.4) ;
4. la comparaison avec le *bagging* et le *boosting* sur spam (figure 15.1) et sur la
   simulation des « sphères emboîtées » (figure 15.2).

Le second notebook traite de l'importance des variables, des proximités et de l'analyse
biais/variance (section 15.4).

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch15_forets as rf
set_style()

## 1. Point de départ : le *bagging*

Rappelons le *bagging* (section 8.7). On dispose d'un échantillon d'apprentissage
$\mathbf Z=\{(x_1,y_1),\dots,(x_N,y_N)\}$. Pour $b=1,\dots,B$ :

- on tire un **échantillon bootstrap** $\mathbf Z^{*b}$ : $N$ observations tirées
  **avec remise** dans $\mathbf Z$ ;
- on ajuste un prédicteur $\hat f^{*b}$ (ici un arbre) sur $\mathbf Z^{*b}$.

Le prédicteur « bagué » est la moyenne $\hat f_{\rm bag}(x)=\frac1B\sum_{b=1}^B \hat f^{*b}(x)$
(en régression), ou le vote majoritaire des $\hat f^{*b}(x)$ (en classification).

> **Intuition.** Un arbre profond a un **faible biais** (il peut épouser des structures
> compliquées) mais une **forte variance** : changer quelques observations modifie les
> premières coupures et donc tout l'arbre. Moyenner beaucoup d'arbres « bruités mais à
> peu près sans biais » réduit la variance.

Les arbres $\hat f^{*1},\dots,\hat f^{*B}$ sont **identiquement distribués** (i.d.) :
ils sont construits par le même procédé aléatoire. Par linéarité de l'espérance,
$$\mathbb E\big[\hat f_{\rm bag}(x)\big]=\frac1B\sum_{b=1}^B\mathbb E\big[\hat f^{*b}(x)\big]=\mathbb E\big[\hat f^{*1}(x)\big],$$
donc **le biais du bagging est celui d'un seul arbre** : le seul gain possible porte sur
la variance. (Le boosting, au contraire, construit des arbres adaptativement pour réduire
le biais ; ses arbres ne sont pas i.d.)

## 2. Variance d'une moyenne de variables corrélées - formule (15.1)

Soient $T_1,\dots,T_B$ des variables aléatoires de même variance $\sigma^2$ et de
corrélation deux à deux $\rho$, c'est-à-dire $\operatorname{Cov}(T_b,T_{b'})=\rho\sigma^2$
pour $b\neq b'$. On développe la variance de la moyenne par bilinéarité de la covariance :
$$
\begin{aligned}
\operatorname{Var}\Big(\frac1B\sum_{b=1}^B T_b\Big)
&=\frac1{B^2}\Big(\sum_{b=1}^B\operatorname{Var}(T_b)+\sum_{b\neq b'}\operatorname{Cov}(T_b,T_{b'})\Big)\\
&=\frac1{B^2}\big(B\sigma^2+B(B-1)\rho\sigma^2\big)
=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2 .
\end{aligned}
$$
C'est la formule (15.1) du livre (la démonstration détaillée et le cas $\rho<0$ font
l'objet de l'exercice 15.1).

> **À retenir.** Quand $B\to\infty$, le second terme disparaît mais le premier,
> $\rho\sigma^2$, **reste** : la corrélation entre les arbres limite le bénéfice de la
> moyenne. Pour aller plus loin il faut **diminuer $\rho$** sans trop augmenter $\sigma^2$.

Vérifions la formule par simulation : on tire des vecteurs gaussiens équicorrélés et on
mesure la variance empirique de leur moyenne.

In [ ]:
rng = np.random.RandomState(0)
sigma2 = 1.0
B_grille = np.arange(1, 51)
rhos = [0.0, 0.05, 0.3, 0.7]
n_rep = 20000

var_emp = {}
for rho in rhos:
    # T_b = sqrt(rho) * U + sqrt(1-rho) * E_b  : variance 1, corrélation rho
    U = rng.randn(n_rep, 1)
    E = rng.randn(n_rep, B_grille.max())
    T = np.sqrt(rho) * U + np.sqrt(1 - rho) * E
    moy_cum = np.cumsum(T, axis=1) / B_grille
    var_emp[rho] = moy_cum.var(axis=0)